In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
from embedder import Embedder
import numpy as np

ember = Embedder()

query = "How does approximate nearest neighbor search work?"


doc = next(d for d in documents if "07-sqlitesearch-vector" in d["filename"])

print(doc["filename"])

v_doc = ember.encode(doc["content"])
v_query = ember.encode(query)

print(v_query.dot(v_doc))


2026-06-21 18:18:09.853136909 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


02-vector-search/lessons/07-sqlitesearch-vector.md
0.36107026789538205


In [8]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [9]:

from tqdm.auto import tqdm

texts = [chunk["content"] for chunk in chunks]
batch_size = 50
X = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    X.extend(ember.encode_batch(batch))

X = np.array(X)
print(X.shape)

  0%|          | 0/6 [00:00<?, ?it/s]

(295, 384)


In [10]:
scores = X.dot(v_query)
best_idx = np.argmax(scores)
print("Best matching chunk content:", chunks[best_idx]["filename"])

Best matching chunk content: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [ ]:
# q4
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [25]:
query = "What metric do we use to evaluate a search engine?"
query_vector = ember.encode(query)

results = vindex.search(query_vector, num_results=5)

for r1 in results:
    print(r1["filename"])

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md


In [26]:
# q5

from minsearch import Index
sindex = Index(text_fields=["content"])
sindex.fit(chunks)

In [29]:
query1 = "How do I store vectors in PostgreSQL?"
results1 = sindex.search(query1, num_results=5)

for r in results1:
    print(r["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [30]:
 # Vector search for same query
 query_vector1 = ember.encode(query1)
 vector_results = vindex.search(query_vector1, num_results=5)
 
 vector_files = set(r["filename"] for r in vector_results)
 text_files = set(r["filename"] for r in results1)
 
 print("In vector but NOT in text:")
 print(vector_files - text_files)


In vector but NOT in text:
{'02-vector-search/lessons/08-pgvector.md'}
